# Carregamento de Bibliotecas e do Dataframe

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Carrega o csv em uma variável to tipo pandas dataframe
df_viagem = pd.read_csv(
    '2026_Viagem.csv', 
    sep=';', 
    decimal=',', 
    encoding='latin-1', 
    parse_dates=['Período - Data de início', 'Período - Data de fim'],
    dayfirst=True  # <-- data no formato brasileiro (dd/mm/yyyy)
)


Primeiramente vamos dar uma olhada geral nas primeiras linhas do dataframe para ter uma compreensão inicial de como estão os dados. 

In [ ]:
df_viagem.head(10)

In [ ]:
df_viagem.info()

# Apresentação da Base de Dados
<p> Esta base foi obtida pelo portal da transparência e trata-se das viagens realizadas por servidores públicos a serviço dos diversos órgãos no ano de 2026, atualizada até dia 16/08/2026. O link para a base encontra-se no README do projeto. </p>

Como pode-se observar pelo df.info() executado anteriormente que tratam-se de 22 colunas a maioria das colunas é auto-explicativa e pouco mais de 406 mil linhas.

Os valores monetários relacionados as quatro últimas colunas são em BRL.

As datas de inicio e fim das viagens estão no formato americano de ano-mes-dia ou também conhecido como yyyy-mm-dd

# Perguntas que gostaríamos de responder

-1 Quais órgãos públicos realizam mais viagens e concentram os maiores gastos? Como o gasto se distribui entre os órgaos e qual o custo médio por viagem ?

-2 Quais características estão associadas às viagens de maior custo? Existem viagens ou viajantes com gastos significativamente acima do padrão ?

-3 Qual é a composição dos gastos nas suas categorias : diárias, passagens, outros gastos, etc.

-4 Qual é a proporção de viagens realizadas vs não realizadas ? Qual o impacto financeiro das viagens não realizadas ? 

-5 Viagens marcadas como urgente apresentam custo médio maior do que viagens não urgentes? Existe correlação entre urgência e custo da viagem ? 

## Valores "Sem informação" e "Sem informaçã":
- Verificar a relevância das colunas para as perguntas a serem respondidas.  
- Não há necessidade de tratamento, se as colunas e esses valores não forem relevantes para as perguntas da análise.

#### Contagem por coluna de registros com valor "Sem informação"

| Coluna | Registros |
|:---|---:|
| Descrição Função | 191.806 |
| Justificativa Urgência Viagem | 112.914 |
| Destinos | 1.206 |

#### Contagem por coluna de registros com valor "Sem informaçã"

| Coluna | Registros |
|:---|---:|
| Número da Proposta (PCDP) | 677 |

In [ ]:
## Em algumas colunas do dataframe constam os valores "Sem informação" e "Sem informaçã". 
## Verificar a relevância das colunas para as perguntas a serem respondidas. 
## Não há necessidade de tratamento, se as colunas e esses valores não forem relevantes para as perguntas da análise.

contagem1 = (df_viagem == "Sem informação").sum()
contagem1 = contagem1[contagem1 > 0].sort_values(ascending=False)

print('Contagem por coluna de registros com valor "Sem informação":')
print(contagem1)

contagem2 = (df_viagem == "Sem informaçã").sum()
contagem2 = contagem2[contagem2 > 0].sort_values(ascending=False)

print('\nContagem por coluna de registros com valor "Sem informaçã":')
print(contagem2)

## Identificação de linhas em que todos os gastos são zero.

- O que fazer com essas linhas com todos gastos zerados?
- Os identificadores do processo de viagem de df_gastos_zerados constam na tabela 2026_Pagamento?
- Se não constarem na tabela pagamentos, essas linhas devem excluídas?

In [ ]:
## O retorno das 5 primeiras linhas de df_viagem mostra que há linhas em que todos os gastos são zero. 

# Criando um filtro para listar somente as colunas de valores monetários:
colunas_gastos = ["Valor diárias", "Valor passagens", "Valor devolução", "Valor outros gastos"]

#Filtrando as linhas onde todos os valores de gastos são zero:
df_gastos_zerados = df_viagem[df_viagem[colunas_gastos].eq(0).all(axis=1)]
print(f'Quantidade de linhas e colunas onde todos os valores de gastos são zero: {df_gastos_zerados.shape[0]} linhas e {df_gastos_zerados.shape[1]} colunas \n')

In [ ]:
df_gastos_zerados.info()

In [ ]:
# O que fazer com essas linhas com todos gastos zerados?
# Os identificadores do processo de viagem de df_gastos_zerados constam na tabela 2026_Pagamento?
# Se não constarem na tabela pagamentos, essas linhas devem excluídas?

#lendo o arquivo 2026_Pagamento
df_pagamento = pd.read_csv('2026_Pagamento.csv', sep=';', decimal=',', encoding='latin-1')
display(df_pagamento.head(3))

In [ ]:
#Checando se na tabela de pagamentos constam os Id's refrente aos registros com todoso os gastos zerados: 
#criando série dos id's de df_gastos_zerados
id_gastos_zerados = df_gastos_zerados['Identificador do processo de viagem']

df_pag_zerados = df_pagamento[
    df_pagamento["Identificador do processo de viagem"].isin(id_gastos_zerados)
]

df_pag_zerados

#Retornou um dataframe vazio. A tabela de pagamentos do Portal da Transparência só registra viagens que geraram desembolso.

In [ ]:
# Checando os id's das duas tabelas podem ser comparados, isto é, são do mesmo tipo:
print(df_viagem["Identificador do processo de viagem"].dtype)
print(df_pagamento["Identificador do processo de viagem"].dtype)

## Resumo Estatístico para as colunas de valores dos gastos:

In [ ]:
# Resumo estatísitco das colunas de gastos:
colunas_gastos = ["Valor diárias", "Valor passagens", "Valor devolução", "Valor outros gastos"]
print(round(df_viagem[colunas_gastos].describe(), 2))

## Prévia de análise de viagens com a situação "Não realizada":

In [ ]:
# Prévia de análise de viagens com a situação "Não realizada"
print(df_viagem["Situação"].value_counts())
print('\n')

# Filtrando não realizadas:
df_nao_realizadas = df_viagem[df_viagem["Situação"]=="Não realizada"]
print('Qtde de linhas e colunas:',df_nao_realizadas.shape)

# Percentual de viagens não realizadas que não constam quaisquer devoluções de valores:
qtd = (df_nao_realizadas['Valor devolução'] == 0).sum()
total = len(df_nao_realizadas)

print(f'Viagens não realizadas sem devolução de valor: {qtd} de {total} ({qtd/total:.1%})')

print('\n')
print("Primeiras cinco linhas de df_nao_realizadas:")
display(df_nao_realizadas.head())

In [ ]:
# Comparando não realizadas com a tabela 2026_Pagamento:

id_nao_realizadas = df_nao_realizadas['Identificador do processo de viagem']

df_pagto_nao_realizadas = df_pagamento[
    df_pagamento["Identificador do processo de viagem"].isin(id_nao_realizadas)
]

df_pagto_nao_realizadas.shape

In [ ]:
resumo = df_pagto_nao_realizadas.groupby('Nome do órgão superior').agg(
    qtd_processos=('Identificador do processo de viagem', 'nunique'),
    qtd_pagamentos=('Identificador do processo de viagem', 'count'),
    total_pago=('Valor', 'sum'),
    media_paga=('Valor', 'mean'),
    maior_pago=('Valor', 'max')
).sort_values('total_pago', ascending=False)

In [ ]:
resumo.loc['TOTAL'] = [
    df_pagto_nao_realizadas['Identificador do processo de viagem'].nunique(),
    df_pagto_nao_realizadas['Identificador do processo de viagem'].count(),
    df_pagto_nao_realizadas['Valor'].sum(),
    df_pagto_nao_realizadas['Valor'].mean(),
    df_pagto_nao_realizadas['Valor'].max()
]

resumo.sort_values('Nome do órgão superior')